In [1]:
import rdkit
from rdkit import Chem
from rdkit.Chem import AllChem, DataStructs #Draw
from rdkit import Chem
from rdkit import Chem
from rdkit.Chem import rdChemReactions

In [4]:
#除补充材料中的63个反应以外，原文还采用了部分并未写在补充材料中的反应，编号从64开始

In [5]:
#判断两个分子是否相同
def equal(a,b):
    return bool(a.GetSubstructMatch(b)) and bool(b.GetSubstructMatch(a))

In [6]:
#算smile分子量
def get_mass(mole):
    mass = 0
    for x in mole.GetAtoms():
        mass+=x.GetMass()
    return mass

In [7]:
#两部分合并，无重复则将分子加入，有重复则把新的生成反应和起始物添加进parents列表
def jiaru(reactants,products):
    for i in products:      
        if get_mass(i.getsmile())<300:
            chongfu= False
            for j in reactants:
                if equal(i.getsmile(),j.getsmile()):
                    chongfu=True
                    j.addparents(i.getparents())
            if not chongfu:
                reactants.append(i)
        else:
            pass
    return reactants

In [13]:
#检查特定小分子是否已经产生
def special_check(reactants):
    count={'O=C':None,'O=CC=C':None,'N#CC#C':None,'N#CC(N)=C(N)C#N':None,'C#N':None,'N':None,'S':None,
           'COO':None,'O=C=O':None,'NC(=N)NC(=N)N':None,'O=CCO':None,'CC#N':None}
    for i in reactants:
        if equal(i.getsmile(),Chem.MolFromSmiles('N')):
            count['N']=i
        if equal(i.getsmile(),Chem.MolFromSmiles('COO')):
            count['COO']=i
        if equal(i.getsmile(),Chem.MolFromSmiles('O=C=O')):
            count['O=C=O']=i
        if equal(i.getsmile(),Chem.MolFromSmiles('S')):
            count['S']=i
        if equal(i.getsmile(),Chem.MolFromSmiles('C#N')):
            count['C#N']=i
        if equal(i.getsmile(),Chem.MolFromSmiles('O=C')):
            count['O=C']=i
        if equal(i.getsmile(),Chem.MolFromSmiles('O=CC=C')):
            count['O=CC=C']=i
        if equal(i.getsmile(),Chem.MolFromSmiles('N#CC#C')):
            count['N#CC#C']=i
        if equal(i.getsmile(),Chem.MolFromSmiles('N#CC(N)=C(N)C#N')):
            count['N#CC(N)=C(N)C#N']=i
        if equal(i.getsmile(),Chem.MolFromSmiles('NC(=N)NC(=N)N')):
            count['NC(=N)NC(=N)N']=i
        if equal(i.getsmile(),Chem.MolFromSmiles('O=CCO')):
            count['O=CCO']=i
        if equal(i.getsmile(),Chem.MolFromSmiles('CC#N')):
            count['CC#N']=i
    return count
#输出的结果记为note，如note_0,note_01,note_012

In [15]:
#统计反应数量
def reaction_count(list):
    count={}
    for i in list:
        if i.getparents()[-1]['reaction'] in count:
            count[i.getparents()[-1]['reaction']]+=1
        else:
            count[i.getparents()[-1]['reaction']]=1
    return count
    


In [19]:
#找特定分子
def find(list,mole):
    find=0
    for i in list:
        if equal(i.getsmile(),Chem.MolFromSmiles(mole)):
            find=i
    return find

In [47]:
#def route(mole):
#    if mole.getlayer()==0:
#        return '(end['+Chem.MolToSmiles(mole.getsmile())+'])'
#    else:
#        routes=''
#        for i in mole.getparents():
#            parents=''
#            for j in i['parents']:
#                parents+=route(j)
#            routes+='('+parents+i['reaction']+')'
#        return '('+routes+'['+Chem.MolToSmiles(mole.getsmile())+'])'

In [49]:
#不能统计重复次数
#def route(mole, current_path=None):
#    # 初始化当前路径集合（仅在首次调用时创建）
#    if current_path is None:
#        current_path = set()
#    
#    # 获取当前分子的SMILES（作为唯一标识）
#    current_smiles = Chem.MolToSmiles(mole.getsmile())
#    
#    # 1. 检查是否出现循环：当前分子已在当前路径中
#    if current_smiles in current_path:
#        return f'(cycle[{current_smiles}])'
#    
#    # 2. 检查是否到达第0层（起始分子）
#    if mole.getlayer() == 0:
#        return f'(end[{current_smiles}])'
#    
#    # 3. 将当前分子加入路径（标记为“已访问”）
#    current_path.add(current_smiles)
#    
#    # 4. 递归遍历所有父分子，构建路径
#    routes = ''
#    for parent_info in mole.getparents():
#        parent_route = ''
#        for parent_mol in parent_info['parents']:
#            # 递归调用，传递当前路径
#            parent_route += route(parent_mol, current_path)
#        # 拼接反应信息
#        routes += f'({parent_route}{parent_info["reaction"]})'
#    
#    # 5. 回溯：将当前分子从路径中移除（让其他路径可以访问）
#    current_path.remove(current_smiles)
#    
#    # 6. 返回最终路径字符串
#    return f'({routes}[{current_smiles}])'

In [51]:
#可以统计重复次数
def route(mole, current_path=None):
    if current_path is None:
        current_path = set()
    
    current_smiles = Chem.MolToSmiles(mole.getsmile())
    if current_smiles in current_path:
        return f'(cycle[{current_smiles}])'
    if mole.getlayer() == 0:
        return f'(end[{current_smiles}])'
    
    current_path.add(current_smiles)
    routes = ''
    # 新增：用字典去重合并相同的反应路径
    unique_reactions = {}
    for parent_info in mole.getparents():
        # 生成每个反应物的路径
        parent_route_parts = []
        for parent_mol in parent_info['parents']:
            parent_route_parts.append(route(parent_mol, current_path))
        parent_route = ''.join(parent_route_parts)
        # 反应的唯一标识：路径+反应编号
        reaction_key = f"{parent_route}{parent_info['reaction']}"
        # 统计相同反应出现的次数
        if reaction_key in unique_reactions:
            unique_reactions[reaction_key] += 1
        else:
            unique_reactions[reaction_key] = 1
    
    # 拼接合并后的路径，重复的标注数量
    for reaction_str, count in unique_reactions.items():
        if count > 1:
            routes += f'({reaction_str})×{count}'
        else:
            routes += f'({reaction_str})'
    
    current_path.remove(current_smiles)
    return f'({routes}[{current_smiles}])'

In [ ]:
glyoxylate={1:'CC(=O)S',2:'OC(=O)CC(O)(CC(O)=O)C(O)=O',3:'OC(=O)CC(C(O)=O)C(O)C(O)=O',4:'O=CC(=O)O',
        5:'OC(=O)CCC(=O)O',6:'OC(=O)CC(O)C(=O)O',7:'OC(=O)C(=O)CC(=O)O',}

calvin={1:'OCC(=O)C(O)C(O)CO',2:'OCC(O)C(=O)O',3:'OCC(O)C=O',4:'OCC(O)=C(O)',5:'OCC=O'}

TCA={1:'CC(=O)S',2:'OC(=O)CC(O)(CC(O)=O)C(O)=O',3:'OC(=O)CC(C(=O)O)=CC(=O)O',4:'OC(=O)CC(C(O)=O)C(O)C(O)=O',5:'OC(=O)CCC(=O)C(=O)O',
     6:'OC(=O)CCC(=O)S',7:'OC(=O)CCC(=O)O',8:'OC(=O)C=CC(=O)O',9:'OC(=O)CC(O)C(=O)O',10:'OC(=O)C(=O)CC(=O)O'}

rTCA={1:'CC(=O)S',2:'CC(=O)C(=O)O',3:'OC(=O)C(=O)CC(=O)O',4:'OC(=O)CC(O)C(=O)O',
      5:'OC(=O)C=CC(=O)O',6:'OC(=O)CCC(=O)O',7:'OC(=O)CCC(=O)S',8:'OC(=O)CCC(=O)C(=O)O',
      9:'OC(=O)CC(C(=O)O)C(=O)C(=O)O',10:'OC(=O)CC(C(O)=O)C(O)C(O)=O',11:'OC(=O)CC(C(=O)O)=CC(=O)O',12:'OC(=O)CC(O)(CC(O)=O)C(O)=O'}

HP={1:'CC(=O)S',2:'OC(=O)CC(=O)S',3:'OCCC(=O)O',4:'CCC(=O)S',5:'CC(C(=O)O)C(=O)S',6:'OC(=O)CCC(=O)S',7:'OC(=O)CCC(=O)O',8:'OC(=O)C=CC(=O)O',
    9:'OC(=O)CC(O)C(=O)O',10:'SC(=O)CC(O)C(=O)O',11:'O=CC(=O)O',12:'OC(=O)C(O)C(C)C(=O)S',13:'OC(=O)C=C(C)C(=O)S',14:'SC(=O)C=C(C)C(=O)O',
    15:'SC(=O)CC(C)(O)C(=O)O',16:'CC(=O)C(=O)O'}

DC_1={1:'CC(=O)S',2:'OC(=O)CC(=O)S',3:'O=CCC(=O)O',4:'OCCC(=O)O',5:'OCCC(=O)S',6:'C=CC(=O)S',
    7:'CCC(=O)S',8:'CC(C(=O)O)C(=O)S',9:'OC(=O)CCC(=O)S',10:'O=CCCC(=O)O',11:'OCCCC(=O)O',12:'OCCCC(=O)S'}

DC_2={1:'CC=CC(=O)S',2:'CC(O)CC(=O)S',3:'CC(=O)CC(=O)S',4:'OCCC(=O)O',5:'OCCC(=O)S',6:'C=CC(=O)S',
    7:'CCC(=O)S',8:'CC(C(=O)O)C(=O)S',9:'OC(=O)CCC(=O)S',10:'O=CCCC(=O)O',11:'OCCCC(=O)O',12:'OCCCC(=O)S'}